In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
fused_df_named_int = pd.read_csv('/content/drive//MyDrive/par-rest.csv')
fused_df_named_int.head()

,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_23,feat_24,feat_25,feat_26,feat_27,feat_28,feat_29,feat_30,feat_31,PD_Label_Combined
0,-0.012186,-0.242798,0.211955,-0.160821,-0.349731,0.206071,-0.035737,0.214398,0.037981,-0.006974,...,0.137540,-0.160455,-0.271169,-0.093879,-0.173949,-0.141305,0.172487,0.109752,-0.024355,1
1,0.087441,-0.137109,0.237290,0.187446,-0.029030,-0.078206,0.246426,-0.176444,-0.205465,-0.180480,...,0.127048,-0.128230,0.238130,0.211441,-0.247213,0.162762,0.005994,-0.129804,0.303167,1
2,0.051393,-0.216597,0.221062,-0.168011,-0.326128,0.184495,-0.004922,0.213458,0.099279,0.024897,...,0.057531,-0.161128,-0.235423,-0.075906,-0.206909,-0.124901,0.233371,0.128156,0.019849,0
3,-0.160189,-0.281906,0.161437,0.190605,-0.135677,-0.167509,-0.101194,-0.114843,-0.063206,0.097344,...,0.302572,-0.225723,-0.163523,0.298178,0.242114,-0.049313,-0.204896,0.152081,-0.115680,1
4,0.117965,0.233021,0.214486,-0.008514,-0.086752,0.019917,-0.214910,-0.170774,0.145858,-0.108719,...,0.215618,0.231745,0.029170,0.208857,-0.181698,0.036961,-0.212452,0.130756,-0.122318,1


In [4]:

# ============================================================
# ENSEMBLE CLASSIFICATION WITH 95% CONFIDENCE INTERVALS
# Stratified 10-Fold CV + Independent Test Set
#
# Models:
#   1. CatBoost
#   2. Gradient Boosting
#   3. Bagging
#   4. AdaBoost
#   5. Extra Trees
#
# Metrics with 95% CI:
# Accuracy, Precision, Sensitivity, Specificity, F1, AUC
#
# Additional metrics:
# Balanced Accuracy, MCC
#
# Notes:
#   - 90% training / 10% independent test split
#   - Stratified 10-fold CV on the training set
#   - Median imputation fitted within each training partition
#   - Wilson CIs for Accuracy, Precision, Sensitivity, Specificity
#   - Stratified bootstrap CIs for F1 and AUC
# ============================================================

!pip install -q catboost scikit-learn statsmodels pandas numpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    GradientBoostingClassifier,
    BaggingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from statsmodels.stats.proportion import proportion_confint
from catboost import CatBoostClassifier


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

feature_cols = [f"feat_{i}" for i in range(32)]

X = fused_df_named_int[feature_cols].copy()
y = fused_df_named_int["PD_Label_Combined"].copy()

X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
y = pd.to_numeric(y, errors="coerce")

valid = y.notna()

X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].astype(int).reset_index(drop=True)

if set(y.unique()) != {0, 1}:
    raise ValueError("Labels must be encoded as 0 and 1.")

if X.shape[1] != 32:
    raise ValueError("Expected exactly 32 embedding features.")

if y.value_counts().min() < 10:
    raise ValueError(
        "Each class needs at least 10 samples for stratified "
        "10-fold cross-validation."
    )

print("X shape:", X.shape)
print("\nClass distribution:")
print(y.value_counts().sort_index())


# ============================================================
# STEP 2: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    stratify=y,
    random_state=42
)

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())

print("\nIndependent test class distribution:")
print(y_test.value_counts().sort_index())


# ============================================================
# STEP 3: MODEL FACTORY
# ============================================================

def create_models(y_fit):

    # Class imbalance ratio for CatBoost.
    n_negative = int((y_fit == 0).sum())
    n_positive = int((y_fit == 1).sum())

    scale_pos_weight = n_negative / max(n_positive, 1)

    # --------------------------------------------------------
    # 1. CatBoost
    # --------------------------------------------------------
    catboost = CatBoostClassifier(
        iterations=500,
        depth=5,
        learning_rate=0.03,
        loss_function="Logloss",
        eval_metric="Logloss",
        l2_leaf_reg=5.0,
        scale_pos_weight=scale_pos_weight,
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1
    )



    # --------------------------------------------------------
    # 2. Bagging
    # --------------------------------------------------------
    # Use a decision tree as the base estimator.
    # The constructor below supports newer scikit-learn.
    bagging = BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=None,
            min_samples_split=5,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42
        ),
        n_estimators=200,
        max_samples=0.85,
        max_features=0.90,
        bootstrap=True,
        bootstrap_features=False,
        n_jobs=-1,
        random_state=42
    )

    # --------------------------------------------------------
    # 3. AdaBoost
    # --------------------------------------------------------
    adaboost = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=1,
            min_samples_leaf=3,
            random_state=42
        ),
        n_estimators=200,
        learning_rate=0.03,
        random_state=42
    )

    # --------------------------------------------------------
    # 4. Extra Trees
    # --------------------------------------------------------
    extra_trees = ExtraTreesClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        bootstrap=False,
        n_jobs=-1,
        random_state=42
    )

    return {
        "CatBoost": catboost,
        "Bagging": bagging,
        "AdaBoost": adaboost,
        "Extra Trees": extra_trees
    }


# ============================================================
# STEP 4: METRICS + 95% CONFIDENCE INTERVALS
# ============================================================

def calculate_metrics_with_ci(
    y_true,
    y_pred,
    y_prob,
    n_boot=2000,
    random_state=42
):

    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    n = len(y_true)

    # ---------------- Point estimates ----------------

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true, y_pred, zero_division=0
    )

    sensitivity = recall_score(
        y_true, y_pred, zero_division=0
    )

    specificity = (
        tn / (tn + fp) if (tn + fp) > 0 else np.nan
    )

    f1 = f1_score(
        y_true, y_pred, zero_division=0
    )

    auc = roc_auc_score(y_true, y_prob)

    results = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "F1": f1,
        "AUC": auc,
        "Balanced Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    # ---------------- Wilson 95% CIs ----------------

    proportions = {
        "Accuracy": (tp + tn, n),
        "Precision": (tp, tp + fp),
        "Sensitivity": (tp, tp + fn),
        "Specificity": (tn, tn + fp)
    }

    for metric, (count, denominator) in proportions.items():

        if denominator > 0:
            low, high = proportion_confint(
                count=count,
                nobs=denominator,
                alpha=0.05,
                method="wilson"
            )
        else:
            low, high = np.nan, np.nan

        results[f"{metric} CI Low"] = low
        results[f"{metric} CI High"] = high

    # ---------------- Stratified bootstrap CIs ----------------
    # Resample each class separately, preserving the class counts.
    # Predictions and probabilities remain paired with labels.

    rng = np.random.default_rng(random_state)

    class_indices = [
        np.flatnonzero(y_true == 0),
        np.flatnonzero(y_true == 1)
    ]

    bootstrap_f1 = []
    bootstrap_auc = []

    for _ in range(n_boot):

        sampled_indices = np.concatenate([
            rng.choice(
                indices,
                size=len(indices),
                replace=True
            )
            for indices in class_indices
        ])

        yt = y_true[sampled_indices]
        yp = y_pred[sampled_indices]
        pr = y_prob[sampled_indices]

        bootstrap_f1.append(
            f1_score(yt, yp, zero_division=0)
        )

        bootstrap_auc.append(
            roc_auc_score(yt, pr)
        )

    for metric, values in [
        ("F1", bootstrap_f1),
        ("AUC", bootstrap_auc)
    ]:

        low, high = np.percentile(
            values,
            [2.5, 97.5]
        )

        results[f"{metric} CI Low"] = low
        results[f"{metric} CI High"] = high

    return results


# ============================================================
# STEP 5: PREPROCESSING
# ============================================================

def make_preprocessor():
    return SimpleImputer(strategy="median")


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CROSS-VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

model_names = [
    "CatBoost",
    "Bagging",
    "AdaBoost",
    "Extra Trees"
]

cv_records = []
cv_summary_records = []
test_records = []
confusion_matrices = {}

metrics_to_report = [
    "Accuracy",
    "Precision",
    "Sensitivity",
    "Specificity",
    "F1",
    "AUC"
]


for model_name in model_names:

    print("\n" + "=" * 85)
    print("EVALUATING:", model_name)
    print("=" * 85)

    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        # Fit imputation only on this fold's training partition.
        imputer = make_preprocessor()

        X_tr_imp = imputer.fit_transform(X_tr)
        X_val_imp = imputer.transform(X_val)

        # Construct a fresh model for each fold.
        models = create_models(y_tr)
        model = models[model_name]

        model.fit(X_tr_imp, y_tr)

        val_pred = np.asarray(
            model.predict(X_val_imp)
        ).reshape(-1).astype(int)

        val_prob = model.predict_proba(X_val_imp)[:, 1]

        metrics = calculate_metrics_with_ci(
            y_val,
            val_pred,
            val_prob,
            n_boot=2000,
            random_state=1000 + fold
        )

        metrics["Model"] = model_name
        metrics["Fold"] = fold

        fold_metrics.append(metrics)
        cv_records.append(metrics.copy())

        # Print metrics and CIs for every fold.
        print(f"\nFold {fold:02d}/10")

        for metric in metrics_to_report:

            print(
                f"{metric:15s}: {metrics[metric]:.4f} "
                f"(95% CI: "
                f"{metrics[metric + ' CI Low']:.4f}-"
                f"{metrics[metric + ' CI High']:.4f})"
            )

    # ========================================================
    # STEP 7: CV MEAN AND STANDARD DEVIATION
    # ========================================================

    fold_df = pd.DataFrame(fold_metrics)

    print(f"\n{model_name} - 10-Fold CV Summary")
    print("-" * 75)

    summary = {"Model": model_name}

    for metric in metrics_to_report:

        mean_val = fold_df[metric].mean()
        std_val = fold_df[metric].std(ddof=1)

        # Descriptive average of the fold CI endpoints.
        # These are NOT a formal pooled 95% CI.
        mean_ci_low = fold_df[
            f"{metric} CI Low"
        ].mean()

        mean_ci_high = fold_df[
            f"{metric} CI High"
        ].mean()

        summary[f"CV {metric} Mean"] = mean_val
        summary[f"CV {metric} SD"] = std_val
        summary[f"Mean Fold {metric} CI Low"] = mean_ci_low
        summary[f"Mean Fold {metric} CI High"] = mean_ci_high

        print(
            f"{metric:15s}: {mean_val:.4f} +/- {std_val:.4f} "
            f"| Mean fold CI: "
            f"{mean_ci_low:.4f}-{mean_ci_high:.4f}"
        )

    for metric in ["Balanced Accuracy", "MCC"]:

        summary[f"CV {metric} Mean"] = fold_df[metric].mean()
        summary[f"CV {metric} SD"] = fold_df[metric].std(ddof=1)

    cv_summary_records.append(summary)

    # ========================================================
    # STEP 8: FINAL MODEL AND INDEPENDENT TEST SET
    # ========================================================

    final_imputer = make_preprocessor()

    X_train_imp = final_imputer.fit_transform(X_train)
    X_test_imp = final_imputer.transform(X_test)

    final_models = create_models(y_train)
    final_model = final_models[model_name]

    final_model.fit(X_train_imp, y_train)

    test_pred = np.asarray(
        final_model.predict(X_test_imp)
    ).reshape(-1).astype(int)

    test_prob = final_model.predict_proba(X_test_imp)[:, 1]

    test_metrics = calculate_metrics_with_ci(
        y_test,
        test_pred,
        test_prob,
        n_boot=5000,
        random_state=2026
    )

    test_metrics["Model"] = model_name
    test_records.append(test_metrics.copy())

    confusion_matrices[model_name] = confusion_matrix(
        y_test,
        test_pred,
        labels=[0, 1]
    )

    print(f"\n{model_name} - Independent Test Set")
    print("-" * 75)

    for metric in metrics_to_report:

        print(
            f"{metric:15s}: {test_metrics[metric]:.4f} "
            f"(95% CI: "
            f"{test_metrics[metric + ' CI Low']:.4f}-"
            f"{test_metrics[metric + ' CI High']:.4f})"
        )

    print(
        f"Balanced Accuracy: "
        f"{test_metrics['Balanced Accuracy']:.4f}"
    )

    print(f"MCC: {test_metrics['MCC']:.4f}")

    print("\nConfusion Matrix (rows=true, columns=predicted):")
    print(confusion_matrices[model_name])


# ============================================================
# STEP 9: INDIVIDUAL FOLD RESULTS WITH 95% CIs
# ============================================================

fold_results_df = pd.DataFrame(cv_records)

fold_columns = [
    "Model", "Fold",
    "Accuracy", "Accuracy CI Low", "Accuracy CI High",
    "Precision", "Precision CI Low", "Precision CI High",
    "Sensitivity", "Sensitivity CI Low", "Sensitivity CI High",
    "Specificity", "Specificity CI Low", "Specificity CI High",
    "F1", "F1 CI Low", "F1 CI High",
    "AUC", "AUC CI Low", "AUC CI High",
    "Balanced Accuracy", "MCC",
    "TN", "FP", "FN", "TP"
]

fold_results_df = fold_results_df[fold_columns]

print("\n" + "=" * 100)
print("ALL MODELS: INDIVIDUAL FOLD RESULTS WITH 95% CIs")
print("=" * 100)

display(
    fold_results_df.sort_values(
        ["Model", "Fold"]
    ).round(4)
)


# ============================================================
# STEP 10: CROSS-VALIDATION SUMMARY
# ============================================================

cv_summary_df = pd.DataFrame(cv_summary_records)

print("\n" + "=" * 100)
print("CROSS-VALIDATION MEAN, SD AND MEAN FOLD CIs")
print("=" * 100)

display(cv_summary_df.round(4))


# ============================================================
# STEP 11: INDEPENDENT TEST RESULTS WITH 95% CIs
# ============================================================

test_results_df = pd.DataFrame(
    test_records
).set_index("Model")

test_columns = [
    "Accuracy", "Accuracy CI Low", "Accuracy CI High",
    "Precision", "Precision CI Low", "Precision CI High",
    "Sensitivity", "Sensitivity CI Low", "Sensitivity CI High",
    "Specificity", "Specificity CI Low", "Specificity CI High",
    "F1", "F1 CI Low", "F1 CI High",
    "AUC", "AUC CI Low", "AUC CI High",
    "Balanced Accuracy", "MCC",
    "TN", "FP", "FN", "TP"
]

test_results_df = test_results_df[test_columns]

print("\n" + "=" * 100)
print("INDEPENDENT TEST SET COMPARISON WITH 95% CIs")
print("=" * 100)

display(
    test_results_df.sort_values(
        by="F1",
        ascending=False
    ).round(4)
)


# ============================================================
# STEP 12: INDIVIDUAL MODEL RESULTS
# ============================================================

print("\n" + "=" * 100)
print("INDIVIDUAL MODEL COMPARISON")
print("=" * 100)

print("\nCross-validation summary:")
display(
    cv_summary_df[
        cv_summary_df["Model"].isin(model_names)
    ].round(4)
)

print("\nIndependent test results:")
display(
    test_results_df.round(4)
)


# ============================================================
# STEP 13: CONFUSION MATRICES FOR ALL MODELS
# ============================================================

print("\n" + "=" * 100)
print("INDEPENDENT TEST CONFUSION MATRICES")
print("=" * 100)

for model_name in model_names:
    print(f"\n{model_name}")
    print("Rows = true labels [HC, PD]")
    print("Columns = predicted labels [HC, PD]")
    print(confusion_matrices[model_name])


# ============================================================
# STEP 14: OPTIONAL SAVE RESULTS AS CSV
# ============================================================

fold_results_df.to_csv(
    "ensemble_fold_results_95CI.csv",
    index=False
)

cv_summary_df.to_csv(
    "ensemble_cv_summary_95CI.csv",
    index=False
)

test_results_df.to_csv(
    "ensemble_independent_test_results_95CI.csv"
)

print("\nSaved result files:")
print("1. ensemble_fold_results_95CI.csv")
print("2. ensemble_cv_summary_95CI.csv")
print("3. ensemble_independent_test_results_95CI.csv")


X shape: (320, 32)

Class distribution:
PD_Label_Combined
0    131
1    189
Name: count, dtype: int64

Training shape: (288, 32)
Test shape: (32, 32)

Training class distribution:
PD_Label_Combined
0    118
1    170
Name: count, dtype: int64

Independent test class distribution:
PD_Label_Combined
0    13
1    19
Name: count, dtype: int64

EVALUATING: CatBoost

Fold 01/10
Accuracy       : 0.7586 (95% CI: 0.5789-0.8778)
Precision      : 0.7778 (95% CI: 0.5479-0.9100)
Sensitivity    : 0.8235 (95% CI: 0.5897-0.9381)
Specificity    : 0.6667 (95% CI: 0.3906-0.8619)
F1             : 0.8000 (95% CI: 0.6471-0.9143)
AUC            : 0.8627 (95% CI: 0.7059-0.9706)

Fold 02/10
Accuracy       : 0.6897 (95% CI: 0.5077-0.8272)
Precision      : 0.7500 (95% CI: 0.5050-0.8982)
Sensitivity    : 0.7059 (95% CI: 0.4687-0.8672)
Specificity    : 0.6667 (95% CI: 0.3906-0.8619)
F1             : 0.7273 (95% CI: 0.5453-0.8824)
AUC            : 0.7647 (95% CI: 0.5636-0.9364)

Fold 03/10
Accuracy       : 0.7241 (9

,Model,Fold,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
20,AdaBoost,1,0.6552,0.4735,0.8006,0.6522,0.4489,0.8119,0.8824,0.6566,...,0.8421,0.7377,0.5441,0.8995,0.6078,0.2622,4,8,2,15
21,AdaBoost,2,0.6897,0.5077,0.8272,0.6818,0.4732,0.8364,0.8824,0.6566,...,0.8650,0.6250,0.4044,0.8236,0.6495,0.3442,5,7,2,15
22,AdaBoost,3,0.5172,0.3443,0.6861,0.5600,0.3707,0.7333,0.8235,0.5897,...,0.7556,0.5564,0.3357,0.7598,0.4534,-0.1330,1,11,3,14
23,AdaBoost,4,0.6552,0.4735,0.8006,0.6522,0.4489,0.8119,0.8824,0.6566,...,0.8421,0.7451,0.5466,0.8947,0.6078,0.2622,4,8,2,15
24,AdaBoost,5,0.6552,0.4735,0.8006,0.6296,0.4423,0.7847,1.0000,0.8157,...,0.8293,0.6275,0.4117,0.8333,0.5833,0.3239,2,10,0,17
25,AdaBoost,6,0.6552,0.4735,0.8006,0.6400,0.4452,0.7975,0.9412,0.7302,...,0.8421,0.6495,0.4460,0.8309,0.5956,0.2731,3,9,1,16
26,AdaBoost,7,0.6552,0.4735,0.8006,0.6522,0.4489,0.8119,0.8824,0.6566,...,0.8423,0.7304,0.5465,0.9069,0.6078,0.2622,4,8,2,15
27,AdaBoost,8,0.5862,0.4074,0.7449,0.5926,0.4073,0.7549,0.9412,0.7302,...,0.7907,0.6838,0.4657,0.8676,0.5123,0.0476,1,11,1,16
28,AdaBoost,9,0.7143,0.5294,0.8475,0.6957,0.4913,0.8440,0.9412,0.7302,...,0.8947,0.7166,0.5160,0.9011,0.6524,0.3887,4,7,1,16
29,AdaBoost,10,0.7143,0.5294,0.8475,0.7143,0.5004,0.8619,0.8824,0.6566,...,0.8947,0.7219,0.5053,0.9064,0.6684,0.3800,5,6,2,15



CROSS-VALIDATION MEAN, SD AND MEAN FOLD CIs


,Model,CV Accuracy Mean,CV Accuracy SD,Mean Fold Accuracy CI Low,Mean Fold Accuracy CI High,CV Precision Mean,CV Precision SD,Mean Fold Precision CI Low,Mean Fold Precision CI High,CV Sensitivity Mean,...,Mean Fold F1 CI Low,Mean Fold F1 CI High,CV AUC Mean,CV AUC SD,Mean Fold AUC CI Low,Mean Fold AUC CI High,CV Balanced Accuracy Mean,CV Balanced Accuracy SD,CV MCC Mean,CV MCC SD
0,CatBoost,0.6665,0.0601,0.4851,0.8087,0.7154,0.0502,0.4796,0.8720,0.7235,...,0.5470,0.8589,0.7487,0.0720,0.5516,0.9112,0.6534,0.0605,0.3103,0.1215
1,Bagging,0.6807,0.0349,0.4984,0.8203,0.7205,0.0320,0.4879,0.8740,0.7529,...,0.5773,0.8674,0.7452,0.0619,0.5478,0.9113,0.6640,0.0385,0.3339,0.0753
2,AdaBoost,0.6498,0.0593,0.4685,0.7956,0.6471,0.0460,0.4477,0.8048,0.9059,...,0.6619,0.8399,0.6794,0.0626,0.4722,0.8624,0.5939,0.0660,0.2411,0.1627
3,Extra Trees,0.6670,0.0600,0.4856,0.8090,0.7430,0.0753,0.4940,0.8907,0.6824,...,0.5237,0.8512,0.7405,0.0789,0.5397,0.9065,0.6628,0.0603,0.3305,0.1245



INDEPENDENT TEST SET COMPARISON WITH 95% CIs


...


INDIVIDUAL MODEL COMPARISON

Cross-validation summary:


,Model,CV Accuracy Mean,CV Accuracy SD,Mean Fold Accuracy CI Low,Mean Fold Accuracy CI High,CV Precision Mean,CV Precision SD,Mean Fold Precision CI Low,Mean Fold Precision CI High,CV Sensitivity Mean,...,Mean Fold F1 CI Low,Mean Fold F1 CI High,CV AUC Mean,CV AUC SD,Mean Fold AUC CI Low,Mean Fold AUC CI High,CV Balanced Accuracy Mean,CV Balanced Accuracy SD,CV MCC Mean,CV MCC SD
0,CatBoost,0.6665,0.0601,0.4851,0.8087,0.7154,0.0502,0.4796,0.8720,0.7235,...,0.5470,0.8589,0.7487,0.0720,0.5516,0.9112,0.6534,0.0605,0.3103,0.1215
1,Bagging,0.6807,0.0349,0.4984,0.8203,0.7205,0.0320,0.4879,0.8740,0.7529,...,0.5773,0.8674,0.7452,0.0619,0.5478,0.9113,0.6640,0.0385,0.3339,0.0753
2,AdaBoost,0.6498,0.0593,0.4685,0.7956,0.6471,0.0460,0.4477,0.8048,0.9059,...,0.6619,0.8399,0.6794,0.0626,0.4722,0.8624,0.5939,0.0660,0.2411,0.1627
3,Extra Trees,0.6670,0.0600,0.4856,0.8090,0.7430,0.0753,0.4940,0.8907,0.6824,...,0.5237,0.8512,0.7405,0.0789,0.5397,0.9065,0.6628,0.0603,0.3305,0.1245



Independent test results:


,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,Sensitivity CI High,Specificity,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
Model,,,,,,,,,,,,,,,,,,,,,
CatBoost,0.6875,0.5143,0.8205,0.7647,0.5274,0.9044,0.6842,0.4601,0.8464,0.6923,...,0.8649,0.6721,0.4696,0.8623,0.6883,0.3706,9,4,6,13
Bagging,0.6562,0.4831,0.7959,0.7000,0.4810,0.8545,0.7368,0.5121,0.8819,0.5385,...,0.8500,0.7004,0.4980,0.8826,0.6377,0.2793,7,6,5,14
AdaBoost,0.5625,0.3933,0.7183,0.5806,0.4077,0.7358,0.9474,0.7536,0.9906,0.0000,...,0.7451,0.6235,0.4190,0.8178,0.4737,-0.1486,0,13,1,18
Extra Trees,0.6875,0.5143,0.8205,0.7647,0.5274,0.9044,0.6842,0.4601,0.8464,0.6923,...,0.8649,0.6964,0.4899,0.8826,0.6883,0.3706,9,4,6,13



INDEPENDENT TEST CONFUSION MATRICES

CatBoost
Rows = true labels [HC, PD]
Columns = predicted labels [HC, PD]
[[ 9  4]
 [ 6 13]]

Bagging
Rows = true labels [HC, PD]
Columns = predicted labels [HC, PD]
[[ 7  6]
 [ 5 14]]

AdaBoost
Rows = true labels [HC, PD]
Columns = predicted labels [HC, PD]
[[ 0 13]
 [ 1 18]]

Extra Trees
Rows = true labels [HC, PD]
Columns = predicted labels [HC, PD]
[[ 9  4]
 [ 6 13]]

Saved result files:
1. ensemble_fold_results_95CI.csv
2. ensemble_cv_summary_95CI.csv
3. ensemble_independent_test_results_95CI.csv


In [5]:

# ============================================================
# CLASSIFICATION WITH 95% CONFIDENCE INTERVALS
# Stratified 10-Fold CV + Independent Test Set
#
# Models:
#   1. LightGBM
#   2. Logistic Regression
#   3. Support Vector Machine (SVM)
#   4. Decision Tree
#
# Metrics with 95% CI:
# Accuracy, Precision, Sensitivity, Specificity, F1, AUC
#
# Additional metrics:
# Balanced Accuracy, MCC, TN, FP, FN, TP
# ============================================================

!pip install -q lightgbm scikit-learn statsmodels pandas numpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from statsmodels.stats.proportion import proportion_confint
from lightgbm import LGBMClassifier


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

feature_cols = [f"feat_{i}" for i in range(32)]

X = fused_df_named_int[feature_cols].copy()
y = fused_df_named_int["PD_Label_Combined"].copy()

X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
y = pd.to_numeric(y, errors="coerce")

valid = y.notna()

X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].astype(int).reset_index(drop=True)

if set(y.unique()) != {0, 1}:
    raise ValueError("Labels must be encoded as 0 and 1.")

if X.shape[1] != 32:
    raise ValueError("Expected exactly 32 embedding features.")

if y.value_counts().min() < 10:
    raise ValueError(
        "Each class needs at least 10 samples for stratified "
        "10-fold cross-validation."
    )

print("X shape:", X.shape)
print("\nClass distribution:")
print(y.value_counts().sort_index())


# ============================================================
# STEP 2: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    stratify=y,
    random_state=42
)

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())

print("\nIndependent test class distribution:")
print(y_test.value_counts().sort_index())


# ============================================================
# STEP 3: MODEL FACTORY
# ============================================================

def create_models(y_fit):

    n_negative = int((y_fit == 0).sum())
    n_positive = int((y_fit == 1).sum())

    # Balanced class weights for LightGBM.
    lightgbm = LGBMClassifier(
        objective="binary",
        n_estimators=200,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=4,
        min_child_samples=10,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=2.0,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    # Logistic Regression: scaling is important.
    logistic_regression = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=1.0,
            penalty="l2",
            solver="liblinear",
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ])

    # SVM: scaling and probability estimates.
    svm = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            C=1.0,
            kernel="rbf",
            gamma="scale",
            probability=True,
            class_weight="balanced",
            random_state=42
        ))
    ])

    # Decision Tree.
    decision_tree = DecisionTreeClassifier(
        criterion="gini",
        max_depth=5,
        min_samples_split=5,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42
    )

    return {
        "LightGBM": lightgbm,
        "Logistic Regression": logistic_regression,
        "SVM": svm,
        "Decision Tree": decision_tree
    }


# ============================================================
# STEP 4: METRICS + 95% CONFIDENCE INTERVALS
# ============================================================

def calculate_metrics_with_ci(
    y_true,
    y_pred,
    y_prob,
    n_boot=2000,
    random_state=42
):

    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    n = len(y_true)

    # ---------------- Point estimates ----------------

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true, y_pred, zero_division=0
    )

    sensitivity = recall_score(
        y_true, y_pred, zero_division=0
    )

    specificity = (
        tn / (tn + fp) if (tn + fp) > 0 else np.nan
    )

    f1 = f1_score(
        y_true, y_pred, zero_division=0
    )

    auc = roc_auc_score(y_true, y_prob)

    results = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "F1": f1,
        "AUC": auc,
        "Balanced Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    # ---------------- Wilson 95% CIs ----------------

    proportions = {
        "Accuracy": (tp + tn, n),
        "Precision": (tp, tp + fp),
        "Sensitivity": (tp, tp + fn),
        "Specificity": (tn, tn + fp)
    }

    for metric, (count, denominator) in proportions.items():

        if denominator > 0:
            low, high = proportion_confint(
                count=count,
                nobs=denominator,
                alpha=0.05,
                method="wilson"
            )
        else:
            low, high = np.nan, np.nan

        results[f"{metric} CI Low"] = low
        results[f"{metric} CI High"] = high

    # ---------------- Stratified bootstrap CIs ----------------
    # Resample within each class, preserving class counts.

    rng = np.random.default_rng(random_state)

    class_indices = [
        np.flatnonzero(y_true == 0),
        np.flatnonzero(y_true == 1)
    ]

    bootstrap_f1 = []
    bootstrap_auc = []

    for _ in range(n_boot):

        sampled_indices = np.concatenate([
            rng.choice(
                indices,
                size=len(indices),
                replace=True
            )
            for indices in class_indices
        ])

        yt = y_true[sampled_indices]
        yp = y_pred[sampled_indices]
        pr = y_prob[sampled_indices]

        bootstrap_f1.append(
            f1_score(yt, yp, zero_division=0)
        )

        bootstrap_auc.append(
            roc_auc_score(yt, pr)
        )

    for metric, values in [
        ("F1", bootstrap_f1),
        ("AUC", bootstrap_auc)
    ]:

        low, high = np.percentile(
            values,
            [2.5, 97.5]
        )

        results[f"{metric} CI Low"] = low
        results[f"{metric} CI High"] = high

    return results


# ============================================================
# STEP 5: PREPROCESSING
# ============================================================

def make_preprocessor():
    return SimpleImputer(strategy="median")


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CROSS-VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

model_names = [
    "LightGBM",
    "Logistic Regression",
    "SVM",
    "Decision Tree"
]

cv_records = []
cv_summary_records = []
test_records = []
confusion_matrices = {}

metrics_to_report = [
    "Accuracy",
    "Precision",
    "Sensitivity",
    "Specificity",
    "F1",
    "AUC"
]


for model_name in model_names:

    print("\n" + "=" * 85)
    print("EVALUATING:", model_name)
    print("=" * 85)

    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        # Imputation is fitted on training data only.
        imputer = make_preprocessor()

        X_tr_imp = imputer.fit_transform(X_tr)
        X_val_imp = imputer.transform(X_val)

        # Fresh model for each fold.
        model = create_models(y_tr)[model_name]

        model.fit(X_tr_imp, y_tr)

        val_pred = np.asarray(
            model.predict(X_val_imp)
        ).reshape(-1).astype(int)

        val_prob = model.predict_proba(X_val_imp)[:, 1]

        metrics = calculate_metrics_with_ci(
            y_val,
            val_pred,
            val_prob,
            n_boot=2000,
            random_state=1000 + fold
        )

        metrics["Model"] = model_name
        metrics["Fold"] = fold

        fold_metrics.append(metrics)
        cv_records.append(metrics.copy())

        print(f"\nFold {fold:02d}/10")

        for metric in metrics_to_report:

            print(
                f"{metric:15s}: {metrics[metric]:.4f} "
                f"(95% CI: "
                f"{metrics[metric + ' CI Low']:.4f}-"
                f"{metrics[metric + ' CI High']:.4f})"
            )

    # ========================================================
    # STEP 7: CV MEAN AND STANDARD DEVIATION
    # ========================================================

    fold_df = pd.DataFrame(fold_metrics)

    print(f"\n{model_name} - 10-Fold CV Summary")
    print("-" * 75)

    summary = {"Model": model_name}

    for metric in metrics_to_report:

        mean_val = fold_df[metric].mean()
        std_val = fold_df[metric].std(ddof=1)

        # Descriptive averages of fold CI endpoints,
        # not a formal pooled 95% confidence interval.
        mean_ci_low = fold_df[
            f"{metric} CI Low"
        ].mean()

        mean_ci_high = fold_df[
            f"{metric} CI High"
        ].mean()

        summary[f"CV {metric} Mean"] = mean_val
        summary[f"CV {metric} SD"] = std_val
        summary[f"Mean Fold {metric} CI Low"] = mean_ci_low
        summary[f"Mean Fold {metric} CI High"] = mean_ci_high

        print(
            f"{metric:15s}: {mean_val:.4f} +/- {std_val:.4f} "
            f"| Mean fold CI: "
            f"{mean_ci_low:.4f}-{mean_ci_high:.4f}"
        )

    for metric in ["Balanced Accuracy", "MCC"]:

        summary[f"CV {metric} Mean"] = fold_df[metric].mean()
        summary[f"CV {metric} SD"] = fold_df[metric].std(ddof=1)

    cv_summary_records.append(summary)

    # ========================================================
    # STEP 8: FINAL MODEL AND INDEPENDENT TEST SET
    # ========================================================

    final_imputer = make_preprocessor()

    X_train_imp = final_imputer.fit_transform(X_train)
    X_test_imp = final_imputer.transform(X_test)

    final_model = create_models(y_train)[model_name]

    final_model.fit(X_train_imp, y_train)

    test_pred = np.asarray(
        final_model.predict(X_test_imp)
    ).reshape(-1).astype(int)

    test_prob = final_model.predict_proba(X_test_imp)[:, 1]

    test_metrics = calculate_metrics_with_ci(
        y_test,
        test_pred,
        test_prob,
        n_boot=5000,
        random_state=2026
    )

    test_metrics["Model"] = model_name
    test_records.append(test_metrics.copy())

    confusion_matrices[model_name] = confusion_matrix(
        y_test,
        test_pred,
        labels=[0, 1]
    )

    print(f"\n{model_name} - Independent Test Set")
    print("-" * 75)

    for metric in metrics_to_report:

        print(
            f"{metric:15s}: {test_metrics[metric]:.4f} "
            f"(95% CI: "
            f"{test_metrics[metric + ' CI Low']:.4f}-"
            f"{test_metrics[metric + ' CI High']:.4f})"
        )

    print(
        f"Balanced Accuracy: "
        f"{test_metrics['Balanced Accuracy']:.4f}"
    )

    print(f"MCC: {test_metrics['MCC']:.4f}")

    print("\nConfusion Matrix (rows=true, columns=predicted):")
    print(confusion_matrices[model_name])


# ============================================================
# STEP 9: INDIVIDUAL FOLD RESULTS WITH 95% CIs
# ============================================================

fold_results_df = pd.DataFrame(cv_records)

fold_columns = [
    "Model", "Fold",
    "Accuracy", "Accuracy CI Low", "Accuracy CI High",
    "Precision", "Precision CI Low", "Precision CI High",
    "Sensitivity", "Sensitivity CI Low", "Sensitivity CI High",
    "Specificity", "Specificity CI Low", "Specificity CI High",
    "F1", "F1 CI Low", "F1 CI High",
    "AUC", "AUC CI Low", "AUC CI High",
    "Balanced Accuracy", "MCC",
    "TN", "FP", "FN", "TP"
]

fold_results_df = fold_results_df[fold_columns]

print("\n" + "=" * 100)
print("ALL MODELS: INDIVIDUAL FOLD RESULTS WITH 95% CIs")
print("=" * 100)

display(
    fold_results_df.sort_values(
        ["Model", "Fold"]
    ).round(4)
)


# ============================================================
# STEP 10: CROSS-VALIDATION SUMMARY
# ============================================================

cv_summary_df = pd.DataFrame(cv_summary_records)

print("\n" + "=" * 100)
print("CROSS-VALIDATION MEAN, SD AND MEAN FOLD CIs")
print("=" * 100)

display(cv_summary_df.round(4))


# ============================================================
# STEP 11: INDEPENDENT TEST RESULTS WITH 95% CIs
# ============================================================

test_results_df = pd.DataFrame(
    test_records
).set_index("Model")

test_columns = [
    "Accuracy", "Accuracy CI Low", "Accuracy CI High",
    "Precision", "Precision CI Low", "Precision CI High",
    "Sensitivity", "Sensitivity CI Low", "Sensitivity CI High",
    "Specificity", "Specificity CI Low", "Specificity CI High",
    "F1", "F1 CI Low", "F1 CI High",
    "AUC", "AUC CI Low", "AUC CI High",
    "Balanced Accuracy", "MCC",
    "TN", "FP", "FN", "TP"
]

test_results_df = test_results_df[test_columns]

print("\n" + "=" * 100)
print("INDEPENDENT TEST SET COMPARISON WITH 95% CIs")
print("=" * 100)

display(
    test_results_df.sort_values(
        by="F1",
        ascending=False
    ).round(4)
)


# ============================================================
# STEP 12: INDIVIDUAL MODEL COMPARISON
# ============================================================

print("\n" + "=" * 100)
print("MODEL COMPARISON")
print("=" * 100)

print("\nCross-validation summary:")
display(cv_summary_df.round(4))

print("\nIndependent test results:")
display(test_results_df.round(4))


# ============================================================
# STEP 13: CONFUSION MATRICES
# ============================================================

print("\n" + "=" * 100)
print("INDEPENDENT TEST CONFUSION MATRICES")
print("=" * 100)

for model_name in model_names:

    print(f"\n{model_name}")
    print("Rows = true labels [HC, PD]")
    print("Columns = predicted labels [HC, PD]")
    print(confusion_matrices[model_name])


# ============================================================
# STEP 14: SAVE RESULTS AS CSV
# ============================================================

fold_results_df.to_csv(
    "classification_fold_results_95CI.csv",
    index=False
)

cv_summary_df.to_csv(
    "classification_cv_summary_95CI.csv",
    index=False
)

test_results_df.to_csv(
    "classification_independent_test_results_95CI.csv"
)

print("\nSaved result files:")
print("1. classification_fold_results_95CI.csv")
print("2. classification_cv_summary_95CI.csv")
print("3. classification_independent_test_results_95CI.csv")


X shape: (320, 32)

Class distribution:
PD_Label_Combined
0    131
1    189
Name: count, dtype: int64

Training shape: (288, 32)
Test shape: (32, 32)

Training class distribution:
PD_Label_Combined
0    118
1    170
Name: count, dtype: int64

Independent test class distribution:
PD_Label_Combined
0    13
1    19
Name: count, dtype: int64

EVALUATING: LightGBM

Fold 01/10
Accuracy       : 0.7241 (95% CI: 0.5428-0.8530)
Precision      : 0.7368 (95% CI: 0.5121-0.8819)
Sensitivity    : 0.8235 (95% CI: 0.5897-0.9381)
Specificity    : 0.5833 (95% CI: 0.3195-0.8067)
F1             : 0.7778 (95% CI: 0.6471-0.8889)
AUC            : 0.8186 (95% CI: 0.6422-0.9510)

Fold 02/10
Accuracy       : 0.7241 (95% CI: 0.5428-0.8530)
Precision      : 0.8000 (95% CI: 0.5481-0.9295)
Sensitivity    : 0.7059 (95% CI: 0.4687-0.8672)
Specificity    : 0.7500 (95% CI: 0.4677-0.9111)
F1             : 0.7500 (95% CI: 0.5712-0.8889)
AUC            : 0.7304 (95% CI: 0.5293-0.9119)

Fold 03/10
Accuracy       : 0.7241 (9

,Model,Fold,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
30,Decision Tree,1,0.6552,0.4735,0.8006,0.8182,0.5230,0.9486,0.5294,0.3096,...,0.8276,0.7500,0.5466,0.9020,0.6814,0.3682,10,2,8,9
31,Decision Tree,2,0.5517,0.3755,0.7159,0.7500,0.4093,0.9285,0.3529,0.1731,...,0.6897,0.7132,0.5074,0.8848,0.5931,0.2053,10,2,11,6
32,Decision Tree,3,0.5517,0.3755,0.7159,0.6250,0.3864,0.8152,0.5882,0.3601,...,0.7647,0.6887,0.4828,0.8653,0.5441,0.0874,6,6,7,10
33,Decision Tree,4,0.7241,0.5428,0.8530,0.9091,0.6226,0.9838,0.5882,0.3601,...,0.8752,0.6936,0.4853,0.8824,0.7525,0.5125,11,1,7,10
34,Decision Tree,5,0.5862,0.4074,0.7449,0.6316,0.4104,0.8085,0.7059,0.4687,...,0.8108,0.6152,0.3995,0.8088,0.5613,0.1270,5,7,5,12
35,Decision Tree,6,0.6552,0.4735,0.8006,0.7059,0.4687,0.8672,0.7059,0.4687,...,0.8571,0.6618,0.4583,0.8431,0.6446,0.2892,7,5,5,12
36,Decision Tree,7,0.5172,0.3443,0.6861,0.6667,0.3542,0.8794,0.3529,0.1731,...,0.6667,0.5858,0.3774,0.7868,0.5515,0.1096,9,3,11,6
37,Decision Tree,8,0.5862,0.4074,0.7449,0.6667,0.4171,0.8482,0.5882,0.3601,...,0.7879,0.6005,0.3922,0.7917,0.5858,0.1691,7,5,7,10
38,Decision Tree,9,0.6071,0.4241,0.7643,0.7500,0.4677,0.9111,0.5294,0.3096,...,0.8000,0.5882,0.3717,0.8075,0.6283,0.2533,8,3,8,9
39,Decision Tree,10,0.5000,0.3263,0.6737,0.5789,0.3628,0.7686,0.6471,0.4130,...,0.7502,0.5936,0.3769,0.7968,0.4599,-0.0839,3,8,6,11



CROSS-VALIDATION MEAN, SD AND MEAN FOLD CIs


,Model,CV Accuracy Mean,CV Accuracy SD,Mean Fold Accuracy CI Low,Mean Fold Accuracy CI High,CV Precision Mean,CV Precision SD,Mean Fold Precision CI Low,Mean Fold Precision CI High,CV Sensitivity Mean,...,Mean Fold F1 CI Low,Mean Fold F1 CI High,CV AUC Mean,CV AUC SD,Mean Fold AUC CI Low,Mean Fold AUC CI High,CV Balanced Accuracy Mean,CV Balanced Accuracy SD,CV MCC Mean,CV MCC SD
0,LightGBM,0.7049,0.0440,0.5232,0.8384,0.7607,0.0444,0.5188,0.9022,0.7353,...,0.5780,0.8814,0.7410,0.0566,0.5412,0.9110,0.6980,0.0454,0.3976,0.0892
1,Logistic Regression,0.6286,0.0585,0.4480,0.7788,0.7050,0.0556,0.4568,0.8704,0.6412,...,0.4755,0.8260,0.6762,0.0786,0.4619,0.8632,0.6259,0.0579,0.2507,0.1151
2,SVM,0.6665,0.0784,0.4859,0.8079,0.7598,0.0871,0.5032,0.9038,0.6471,...,0.5006,0.8489,0.7249,0.0898,0.5213,0.8932,0.6705,0.0797,0.3433,0.1578
3,Decision Tree,0.5935,0.0691,0.4150,0.7500,0.7102,0.0994,0.4422,0.8759,0.5588,...,0.4008,0.7830,0.6491,0.0599,0.4398,0.8369,0.6002,0.0813,0.2038,0.1648



INDEPENDENT TEST SET COMPARISON WITH 95% CIs


...


MODEL COMPARISON

Cross-validation summary:


,Model,CV Accuracy Mean,CV Accuracy SD,Mean Fold Accuracy CI Low,Mean Fold Accuracy CI High,CV Precision Mean,CV Precision SD,Mean Fold Precision CI Low,Mean Fold Precision CI High,CV Sensitivity Mean,...,Mean Fold F1 CI Low,Mean Fold F1 CI High,CV AUC Mean,CV AUC SD,Mean Fold AUC CI Low,Mean Fold AUC CI High,CV Balanced Accuracy Mean,CV Balanced Accuracy SD,CV MCC Mean,CV MCC SD
0,LightGBM,0.7049,0.0440,0.5232,0.8384,0.7607,0.0444,0.5188,0.9022,0.7353,...,0.5780,0.8814,0.7410,0.0566,0.5412,0.9110,0.6980,0.0454,0.3976,0.0892
1,Logistic Regression,0.6286,0.0585,0.4480,0.7788,0.7050,0.0556,0.4568,0.8704,0.6412,...,0.4755,0.8260,0.6762,0.0786,0.4619,0.8632,0.6259,0.0579,0.2507,0.1151
2,SVM,0.6665,0.0784,0.4859,0.8079,0.7598,0.0871,0.5032,0.9038,0.6471,...,0.5006,0.8489,0.7249,0.0898,0.5213,0.8932,0.6705,0.0797,0.3433,0.1578
3,Decision Tree,0.5935,0.0691,0.4150,0.7500,0.7102,0.0994,0.4422,0.8759,0.5588,...,0.4008,0.7830,0.6491,0.0599,0.4398,0.8369,0.6002,0.0813,0.2038,0.1648



Independent test results:


,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,Sensitivity CI High,Specificity,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
Model,,,,,,,,,,,,,,,,,,,,,
LightGBM,0.6562,0.4831,0.7959,0.7857,0.5241,0.9243,0.5789,0.3628,0.7686,0.7692,...,0.8333,0.6761,0.4615,0.8704,0.6741,0.3447,10,3,8,11
Logistic Regression,0.5625,0.3933,0.7183,0.6316,0.4104,0.8085,0.6316,0.4104,0.8085,0.4615,...,0.7778,0.5992,0.3846,0.8057,0.5466,0.0931,6,7,7,12
SVM,0.6562,0.4831,0.7959,0.7500,0.5050,0.8982,0.6316,0.4104,0.8085,0.6923,...,0.8421,0.6397,0.4413,0.8340,0.6619,0.3181,9,4,7,12
Decision Tree,0.5938,0.4226,0.7448,0.6667,0.4375,0.8372,0.6316,0.4104,0.8085,0.5385,...,0.7909,0.5931,0.3927,0.7814,0.5850,0.1683,7,6,7,12



INDEPENDENT TEST CONFUSION MATRICES

LightGBM
Rows = true labels [HC, PD]
Columns = predicted labels [HC, PD]
[[10  3]
 [ 8 11]]

Logistic Regression
Rows = true labels [HC, PD]
Columns = predicted labels [HC, PD]
[[ 6  7]
 [ 7 12]]

SVM
Rows = true labels [HC, PD]
Columns = predicted labels [HC, PD]
[[ 9  4]
 [ 7 12]]

Decision Tree
Rows = true labels [HC, PD]
Columns = predicted labels [HC, PD]
[[ 7  6]
 [ 7 12]]

Saved result files:
1. classification_fold_results_95CI.csv
2. classification_cv_summary_95CI.csv
3. classification_independent_test_results_95CI.csv


In [6]:
# ============================================================
# CLASSIFICATION WITH 95% CONFIDENCE INTERVALS
# Stratified 10-Fold CV + Independent Test Set
#
# Models:
#   1. LightGBM
#   2. Logistic Regression
#   3. Support Vector Machine (SVM)
#   4. Decision Tree
#   5. Gaussian Naive Bayes
#   6. Linear Discriminant Analysis (LDA)
#
# Metrics with 95% CI:
# Accuracy, Precision, Sensitivity, Specificity, F1, AUC
#
# Additional metrics:
# Balanced Accuracy, MCC, TN, FP, FN, TP
# ============================================================

!pip install -q lightgbm scikit-learn statsmodels pandas numpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from statsmodels.stats.proportion import proportion_confint
from lightgbm import LGBMClassifier


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

feature_cols = [f"feat_{i}" for i in range(32)]

X = fused_df_named_int[feature_cols].copy()
y = fused_df_named_int["PD_Label_Combined"].copy()

X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
y = pd.to_numeric(y, errors="coerce")

valid = y.notna()

X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].astype(int).reset_index(drop=True)

if set(y.unique()) != {0, 1}:
    raise ValueError("Labels must be encoded as 0 and 1.")

if X.shape[1] != 32:
    raise ValueError("Expected exactly 32 embedding features.")

if y.value_counts().min() < 10:
    raise ValueError(
        "Each class needs at least 10 samples for stratified "
        "10-fold cross-validation."
    )

print("X shape:", X.shape)
print("\nClass distribution:")
print(y.value_counts().sort_index())


# ============================================================
# STEP 2: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    stratify=y,
    random_state=42
)

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())

print("\nIndependent test class distribution:")
print(y_test.value_counts().sort_index())


# ============================================================
# STEP 3: MODEL FACTORY
# ============================================================

def create_models(y_fit):

    # --------------------------------------------------------
    # 1. LightGBM
    # --------------------------------------------------------

    lightgbm = LGBMClassifier(
        objective="binary",
        n_estimators=200,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=4,
        min_child_samples=10,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=2.0,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    # --------------------------------------------------------
    # 2. Logistic Regression
    # Scaling is performed inside each fold.
    # --------------------------------------------------------

    logistic_regression = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=1.0,
            penalty="l2",
            solver="liblinear",
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ])

    # --------------------------------------------------------
    # 3. Support Vector Machine (RBF kernel)
    # Scaling and probability estimates are included.
    # --------------------------------------------------------

    svm = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            C=1.0,
            kernel="rbf",
            gamma="scale",
            probability=True,
            class_weight="balanced",
            random_state=42
        ))
    ])

    # --------------------------------------------------------
    # 4. Decision Tree
    # --------------------------------------------------------

    decision_tree = DecisionTreeClassifier(
        criterion="gini",
        max_depth=5,
        min_samples_split=5,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42
    )

    # --------------------------------------------------------
    # 5. Gaussian Naive Bayes
    # Models each feature using a Gaussian distribution
    # conditional on the class.
    # --------------------------------------------------------

    gaussian_nb = GaussianNB()

    # --------------------------------------------------------
    # 6. Linear Discriminant Analysis (LDA)
    # Shrinkage helps stabilize covariance estimation when
    # features are correlated or the sample size is limited.
    # --------------------------------------------------------

    lda = LinearDiscriminantAnalysis(
        solver="lsqr",
        shrinkage="auto"
    )

    return {
        "LightGBM": lightgbm,
        "Logistic Regression": logistic_regression,
        "SVM": svm,
        "Decision Tree": decision_tree,
        "Gaussian Naive Bayes": gaussian_nb,
        "LDA": lda
    }


# ============================================================
# STEP 4: METRICS + 95% CONFIDENCE INTERVALS
# ============================================================

def calculate_metrics_with_ci(
    y_true,
    y_pred,
    y_prob,
    n_boot=2000,
    random_state=42
):

    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    n = len(y_true)

    # ---------------- Point estimates ----------------

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true, y_pred, zero_division=0
    )

    sensitivity = recall_score(
        y_true, y_pred, zero_division=0
    )

    specificity = (
        tn / (tn + fp) if (tn + fp) > 0 else np.nan
    )

    f1 = f1_score(
        y_true, y_pred, zero_division=0
    )

    auc = roc_auc_score(y_true, y_prob)

    results = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "F1": f1,
        "AUC": auc,
        "Balanced Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    # ---------------- Wilson 95% CIs ----------------

    proportions = {
        "Accuracy": (tp + tn, n),
        "Precision": (tp, tp + fp),
        "Sensitivity": (tp, tp + fn),
        "Specificity": (tn, tn + fp)
    }

    for metric, (count, denominator) in proportions.items():

        if denominator > 0:
            low, high = proportion_confint(
                count=count,
                nobs=denominator,
                alpha=0.05,
                method="wilson"
            )
        else:
            low, high = np.nan, np.nan

        results[f"{metric} CI Low"] = low
        results[f"{metric} CI High"] = high

    # ---------------- Stratified bootstrap CIs ----------------
    # Resample within each class, preserving class counts.

    rng = np.random.default_rng(random_state)

    class_indices = [
        np.flatnonzero(y_true == 0),
        np.flatnonzero(y_true == 1)
    ]

    bootstrap_f1 = []
    bootstrap_auc = []

    for _ in range(n_boot):

        sampled_indices = np.concatenate([
            rng.choice(
                indices,
                size=len(indices),
                replace=True
            )
            for indices in class_indices
        ])

        yt = y_true[sampled_indices]
        yp = y_pred[sampled_indices]
        pr = y_prob[sampled_indices]

        bootstrap_f1.append(
            f1_score(yt, yp, zero_division=0)
        )

        bootstrap_auc.append(
            roc_auc_score(yt, pr)
        )

    for metric, values in [
        ("F1", bootstrap_f1),
        ("AUC", bootstrap_auc)
    ]:

        low, high = np.percentile(
            values,
            [2.5, 97.5]
        )

        results[f"{metric} CI Low"] = low
        results[f"{metric} CI High"] = high

    return results


# ============================================================
# STEP 5: PREPROCESSING
# ============================================================

def make_preprocessor():
    return SimpleImputer(strategy="median")


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CROSS-VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

model_names = [
    "LightGBM",
    "Logistic Regression",
    "SVM",
    "Decision Tree",
    "Gaussian Naive Bayes",
    "LDA"
]

cv_records = []
cv_summary_records = []
test_records = []
confusion_matrices = {}

metrics_to_report = [
    "Accuracy",
    "Precision",
    "Sensitivity",
    "Specificity",
    "F1",
    "AUC"
]


for model_name in model_names:

    print("\n" + "=" * 85)
    print("EVALUATING:", model_name)
    print("=" * 85)

    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        # Imputation is fitted on training data only.
        imputer = make_preprocessor()

        X_tr_imp = imputer.fit_transform(X_tr)
        X_val_imp = imputer.transform(X_val)

        # Create a fresh model for every fold.
        model = create_models(y_tr)[model_name]

        model.fit(X_tr_imp, y_tr)

        val_pred = np.asarray(
            model.predict(X_val_imp)
        ).reshape(-1).astype(int)

        val_prob = model.predict_proba(X_val_imp)[:, 1]

        metrics = calculate_metrics_with_ci(
            y_val,
            val_pred,
            val_prob,
            n_boot=2000,
            random_state=1000 + fold
        )

        metrics["Model"] = model_name
        metrics["Fold"] = fold

        fold_metrics.append(metrics)
        cv_records.append(metrics.copy())

        print(f"\nFold {fold:02d}/10")

        for metric in metrics_to_report:

            print(
                f"{metric:15s}: {metrics[metric]:.4f} "
                f"(95% CI: "
                f"{metrics[metric + ' CI Low']:.4f}-"
                f"{metrics[metric + ' CI High']:.4f})"
            )

    # ========================================================
    # STEP 7: CV MEAN AND STANDARD DEVIATION
    # ========================================================

    fold_df = pd.DataFrame(fold_metrics)

    print(f"\n{model_name} - 10-Fold CV Summary")
    print("-" * 75)

    summary = {"Model": model_name}

    for metric in metrics_to_report:

        mean_val = fold_df[metric].mean()
        std_val = fold_df[metric].std(ddof=1)

        # Descriptive averages of fold CI endpoints.
        # These are not formal pooled 95% confidence intervals.
        mean_ci_low = fold_df[
            f"{metric} CI Low"
        ].mean()

        mean_ci_high = fold_df[
            f"{metric} CI High"
        ].mean()

        summary[f"CV {metric} Mean"] = mean_val
        summary[f"CV {metric} SD"] = std_val
        summary[f"Mean Fold {metric} CI Low"] = mean_ci_low
        summary[f"Mean Fold {metric} CI High"] = mean_ci_high

        print(
            f"{metric:15s}: {mean_val:.4f} +/- {std_val:.4f} "
            f"| Mean fold CI: "
            f"{mean_ci_low:.4f}-{mean_ci_high:.4f}"
        )

    for metric in ["Balanced Accuracy", "MCC"]:

        summary[f"CV {metric} Mean"] = fold_df[metric].mean()
        summary[f"CV {metric} SD"] = fold_df[metric].std(ddof=1)

    cv_summary_records.append(summary)

    # ========================================================
    # STEP 8: FINAL MODEL AND INDEPENDENT TEST SET
    # ========================================================

    final_imputer = make_preprocessor()

    X_train_imp = final_imputer.fit_transform(X_train)
    X_test_imp = final_imputer.transform(X_test)

    final_model = create_models(y_train)[model_name]

    final_model.fit(X_train_imp, y_train)

    test_pred = np.asarray(
        final_model.predict(X_test_imp)
    ).reshape(-1).astype(int)

    test_prob = final_model.predict_proba(X_test_imp)[:, 1]

    test_metrics = calculate_metrics_with_ci(
        y_test,
        test_pred,
        test_prob,
        n_boot=5000,
        random_state=2026
    )

    test_metrics["Model"] = model_name
    test_records.append(test_metrics.copy())

    confusion_matrices[model_name] = confusion_matrix(
        y_test,
        test_pred,
        labels=[0, 1]
    )

    print(f"\n{model_name} - Independent Test Set")
    print("-" * 75)

    for metric in metrics_to_report:

        print(
            f"{metric:15s}: {test_metrics[metric]:.4f} "
            f"(95% CI: "
            f"{test_metrics[metric + ' CI Low']:.4f}-"
            f"{test_metrics[metric + ' CI High']:.4f})"
        )

    print(
        f"Balanced Accuracy: "
        f"{test_metrics['Balanced Accuracy']:.4f}"
    )

    print(f"MCC: {test_metrics['MCC']:.4f}")

    print("\nConfusion Matrix (rows=true, columns=predicted):")
    print(confusion_matrices[model_name])


# ============================================================
# STEP 9: INDIVIDUAL FOLD RESULTS WITH 95% CIs
# ============================================================

fold_results_df = pd.DataFrame(cv_records)

fold_columns = [
    "Model", "Fold",
    "Accuracy", "Accuracy CI Low", "Accuracy CI High",
    "Precision", "Precision CI Low", "Precision CI High",
    "Sensitivity", "Sensitivity CI Low", "Sensitivity CI High",
    "Specificity", "Specificity CI Low", "Specificity CI High",
    "F1", "F1 CI Low", "F1 CI High",
    "AUC", "AUC CI Low", "AUC CI High",
    "Balanced Accuracy", "MCC",
    "TN", "FP", "FN", "TP"
]

fold_results_df = fold_results_df[fold_columns]

print("\n" + "=" * 100)
print("ALL MODELS: INDIVIDUAL FOLD RESULTS WITH 95% CIs")
print("=" * 100)

display(
    fold_results_df.sort_values(
        ["Model", "Fold"]
    ).round(4)
)


# ============================================================
# STEP 10: CROSS-VALIDATION SUMMARY
# ============================================================

cv_summary_df = pd.DataFrame(cv_summary_records)

print("\n" + "=" * 100)
print("CROSS-VALIDATION MEAN, SD AND MEAN FOLD CIs")
print("=" * 100)

display(cv_summary_df.round(4))


# ============================================================
# STEP 11: INDEPENDENT TEST RESULTS WITH 95% CIs
# ============================================================

test_results_df = pd.DataFrame(
    test_records
).set_index("Model")

test_columns = [
    "Accuracy", "Accuracy CI Low", "Accuracy CI High",
    "Precision", "Precision CI Low", "Precision CI High",
    "Sensitivity", "Sensitivity CI Low", "Sensitivity CI High",
    "Specificity", "Specificity CI Low", "Specificity CI High",
    "F1", "F1 CI Low", "F1 CI High",
    "AUC", "AUC CI Low", "AUC CI High",
    "Balanced Accuracy", "MCC",
    "TN", "FP", "FN", "TP"
]

test_results_df = test_results_df[test_columns]

print("\n" + "=" * 100)
print("INDEPENDENT TEST SET COMPARISON WITH 95% CIs")
print("=" * 100)

display(
    test_results_df.sort_values(
        by="F1",
        ascending=False
    ).round(4)
)


# ============================================================
# STEP 12: INDIVIDUAL MODEL COMPARISON
# ============================================================

print("\n" + "=" * 100)
print("MODEL COMPARISON")
print("=" * 100)

print("\nCross-validation summary:")
display(cv_summary_df.round(4))

print("\nIndependent test results:")
display(test_results_df.round(4))


# ============================================================
# STEP 13: CONFUSION MATRICES
# ============================================================

print("\n" + "=" * 100)
print("INDEPENDENT TEST CONFUSION MATRICES")
print("=" * 100)

for model_name in model_names:

    print(f"\n{model_name}")
    print("Rows = true labels [HC, PD]")
    print("Columns = predicted labels [HC, PD]")
    print(confusion_matrices[model_name])


# ============================================================
# STEP 14: SAVE RESULTS AS CSV
# ============================================================

fold_results_df.to_csv(
    "classification_fold_results_95CI.csv",
    index=False
)

cv_summary_df.to_csv(
    "classification_cv_summary_95CI.csv",
    index=False
)

test_results_df.to_csv(
    "classification_independent_test_results_95CI.csv"
)

print("\nSaved result files:")
print("1. classification_fold_results_95CI.csv")
print("2. classification_cv_summary_95CI.csv")
print("3. classification_independent_test_results_95CI.csv")

X shape: (320, 32)

Class distribution:
PD_Label_Combined
0    131
1    189
Name: count, dtype: int64

Training shape: (288, 32)
Test shape: (32, 32)

Training class distribution:
PD_Label_Combined
0    118
1    170
Name: count, dtype: int64

Independent test class distribution:
PD_Label_Combined
0    13
1    19
Name: count, dtype: int64

EVALUATING: LightGBM

Fold 01/10
Accuracy       : 0.7241 (95% CI: 0.5428-0.8530)
Precision      : 0.7368 (95% CI: 0.5121-0.8819)
Sensitivity    : 0.8235 (95% CI: 0.5897-0.9381)
Specificity    : 0.5833 (95% CI: 0.3195-0.8067)
F1             : 0.7778 (95% CI: 0.6471-0.8889)
AUC            : 0.8186 (95% CI: 0.6422-0.9510)

Fold 02/10
Accuracy       : 0.7241 (95% CI: 0.5428-0.8530)
Precision      : 0.8000 (95% CI: 0.5481-0.9295)
Sensitivity    : 0.7059 (95% CI: 0.4687-0.8672)
Specificity    : 0.7500 (95% CI: 0.4677-0.9111)
F1             : 0.7500 (95% CI: 0.5712-0.8889)
AUC            : 0.7304 (95% CI: 0.5293-0.9119)

Fold 03/10
Accuracy       : 0.7241 (9

,Model,Fold,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
30,Decision Tree,1,0.6552,0.4735,0.8006,0.8182,0.5230,0.9486,0.5294,0.3096,...,0.8276,0.7500,0.5466,0.9020,0.6814,0.3682,10,2,8,9
31,Decision Tree,2,0.5517,0.3755,0.7159,0.7500,0.4093,0.9285,0.3529,0.1731,...,0.6897,0.7132,0.5074,0.8848,0.5931,0.2053,10,2,11,6
32,Decision Tree,3,0.5517,0.3755,0.7159,0.6250,0.3864,0.8152,0.5882,0.3601,...,0.7647,0.6887,0.4828,0.8653,0.5441,0.0874,6,6,7,10
33,Decision Tree,4,0.7241,0.5428,0.8530,0.9091,0.6226,0.9838,0.5882,0.3601,...,0.8752,0.6936,0.4853,0.8824,0.7525,0.5125,11,1,7,10
34,Decision Tree,5,0.5862,0.4074,0.7449,0.6316,0.4104,0.8085,0.7059,0.4687,...,0.8108,0.6152,0.3995,0.8088,0.5613,0.1270,5,7,5,12
35,Decision Tree,6,0.6552,0.4735,0.8006,0.7059,0.4687,0.8672,0.7059,0.4687,...,0.8571,0.6618,0.4583,0.8431,0.6446,0.2892,7,5,5,12
36,Decision Tree,7,0.5172,0.3443,0.6861,0.6667,0.3542,0.8794,0.3529,0.1731,...,0.6667,0.5858,0.3774,0.7868,0.5515,0.1096,9,3,11,6
37,Decision Tree,8,0.5862,0.4074,0.7449,0.6667,0.4171,0.8482,0.5882,0.3601,...,0.7879,0.6005,0.3922,0.7917,0.5858,0.1691,7,5,7,10
38,Decision Tree,9,0.6071,0.4241,0.7643,0.7500,0.4677,0.9111,0.5294,0.3096,...,0.8000,0.5882,0.3717,0.8075,0.6283,0.2533,8,3,8,9
39,Decision Tree,10,0.5000,0.3263,0.6737,0.5789,0.3628,0.7686,0.6471,0.4130,...,0.7502,0.5936,0.3769,0.7968,0.4599,-0.0839,3,8,6,11



CROSS-VALIDATION MEAN, SD AND MEAN FOLD CIs


,Model,CV Accuracy Mean,CV Accuracy SD,Mean Fold Accuracy CI Low,Mean Fold Accuracy CI High,CV Precision Mean,CV Precision SD,Mean Fold Precision CI Low,Mean Fold Precision CI High,CV Sensitivity Mean,...,Mean Fold F1 CI Low,Mean Fold F1 CI High,CV AUC Mean,CV AUC SD,Mean Fold AUC CI Low,Mean Fold AUC CI High,CV Balanced Accuracy Mean,CV Balanced Accuracy SD,CV MCC Mean,CV MCC SD
0,LightGBM,0.7049,0.0440,0.5232,0.8384,0.7607,0.0444,0.5188,0.9022,0.7353,...,0.5780,0.8814,0.7410,0.0566,0.5412,0.9110,0.6980,0.0454,0.3976,0.0892
1,Logistic Regression,0.6286,0.0585,0.4480,0.7788,0.7050,0.0556,0.4568,0.8704,0.6412,...,0.4755,0.8260,0.6762,0.0786,0.4619,0.8632,0.6259,0.0579,0.2507,0.1151
2,SVM,0.6665,0.0784,0.4859,0.8079,0.7598,0.0871,0.5032,0.9038,0.6471,...,0.5006,0.8489,0.7249,0.0898,0.5213,0.8932,0.6705,0.0797,0.3433,0.1578
3,Decision Tree,0.5935,0.0691,0.4150,0.7500,0.7102,0.0994,0.4422,0.8759,0.5588,...,0.4008,0.7830,0.6491,0.0599,0.4398,0.8369,0.6002,0.0813,0.2038,0.1648
4,Gaussian Naive Bayes,0.6180,0.0849,0.4390,0.7692,0.6965,0.0867,0.4515,0.8614,0.6412,...,0.4733,0.8185,0.6351,0.0935,0.4183,0.8306,0.6126,0.0931,0.2245,0.1839
5,LDA,0.6390,0.0942,0.4599,0.7854,0.6754,0.0679,0.4511,0.8396,0.7471,...,0.5482,0.8391,0.6859,0.0815,0.4769,0.8681,0.6160,0.0922,0.2447,0.1980



INDEPENDENT TEST SET COMPARISON WITH 95% CIs


...


MODEL COMPARISON

Cross-validation summary:


,Model,CV Accuracy Mean,CV Accuracy SD,Mean Fold Accuracy CI Low,Mean Fold Accuracy CI High,CV Precision Mean,CV Precision SD,Mean Fold Precision CI Low,Mean Fold Precision CI High,CV Sensitivity Mean,...,Mean Fold F1 CI Low,Mean Fold F1 CI High,CV AUC Mean,CV AUC SD,Mean Fold AUC CI Low,Mean Fold AUC CI High,CV Balanced Accuracy Mean,CV Balanced Accuracy SD,CV MCC Mean,CV MCC SD
0,LightGBM,0.7049,0.0440,0.5232,0.8384,0.7607,0.0444,0.5188,0.9022,0.7353,...,0.5780,0.8814,0.7410,0.0566,0.5412,0.9110,0.6980,0.0454,0.3976,0.0892
1,Logistic Regression,0.6286,0.0585,0.4480,0.7788,0.7050,0.0556,0.4568,0.8704,0.6412,...,0.4755,0.8260,0.6762,0.0786,0.4619,0.8632,0.6259,0.0579,0.2507,0.1151
2,SVM,0.6665,0.0784,0.4859,0.8079,0.7598,0.0871,0.5032,0.9038,0.6471,...,0.5006,0.8489,0.7249,0.0898,0.5213,0.8932,0.6705,0.0797,0.3433,0.1578
3,Decision Tree,0.5935,0.0691,0.4150,0.7500,0.7102,0.0994,0.4422,0.8759,0.5588,...,0.4008,0.7830,0.6491,0.0599,0.4398,0.8369,0.6002,0.0813,0.2038,0.1648
4,Gaussian Naive Bayes,0.6180,0.0849,0.4390,0.7692,0.6965,0.0867,0.4515,0.8614,0.6412,...,0.4733,0.8185,0.6351,0.0935,0.4183,0.8306,0.6126,0.0931,0.2245,0.1839
5,LDA,0.6390,0.0942,0.4599,0.7854,0.6754,0.0679,0.4511,0.8396,0.7471,...,0.5482,0.8391,0.6859,0.0815,0.4769,0.8681,0.6160,0.0922,0.2447,0.1980



Independent test results:


,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,Sensitivity CI High,Specificity,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
Model,,,,,,,,,,,,,,,,,,,,,
LightGBM,0.6562,0.4831,0.7959,0.7857,0.5241,0.9243,0.5789,0.3628,0.7686,0.7692,...,0.8333,0.6761,0.4615,0.8704,0.6741,0.3447,10,3,8,11
Logistic Regression,0.5625,0.3933,0.7183,0.6316,0.4104,0.8085,0.6316,0.4104,0.8085,0.4615,...,0.7778,0.5992,0.3846,0.8057,0.5466,0.0931,6,7,7,12
SVM,0.6562,0.4831,0.7959,0.7500,0.5050,0.8982,0.6316,0.4104,0.8085,0.6923,...,0.8421,0.6397,0.4413,0.8340,0.6619,0.3181,9,4,7,12
Decision Tree,0.5938,0.4226,0.7448,0.6667,0.4375,0.8372,0.6316,0.4104,0.8085,0.5385,...,0.7909,0.5931,0.3927,0.7814,0.5850,0.1683,7,6,7,12
Gaussian Naive Bayes,0.6875,0.5143,0.8205,0.8000,0.5481,0.9295,0.6316,0.4104,0.8085,0.7692,...,0.8571,0.7490,0.5628,0.9028,0.7004,0.3945,10,3,7,12
LDA,0.5625,0.3933,0.7183,0.6316,0.4104,0.8085,0.6316,0.4104,0.8085,0.4615,...,0.7778,0.6032,0.3887,0.8058,0.5466,0.0931,6,7,7,12



INDEPENDENT TEST CONFUSION MATRICES

LightGBM
Rows = true labels [HC, PD]
Columns = predicted labels [HC, PD]
[[10  3]
 [ 8 11]]

Logistic Regression
Rows = true labels [HC, PD]
Columns = predicted labels [HC, PD]
[[ 6  7]
 [ 7 12]]

SVM
Rows = true labels [HC, PD]
Columns = predicted labels [HC, PD]
[[ 9  4]
 [ 7 12]]

Decision Tree
Rows = true labels [HC, PD]
Columns = predicted labels [HC, PD]
[[ 7  6]
 [ 7 12]]

Gaussian Naive Bayes
Rows = true labels [HC, PD]
Columns = predicted labels [HC, PD]
[[10  3]
 [ 7 12]]

LDA
Rows = true labels [HC, PD]
Columns = predicted labels [HC, PD]
[[ 6  7]
 [ 7 12]]

Saved result files:
1. classification_fold_results_95CI.csv
2. classification_cv_summary_95CI.csv
3. classification_independent_test_results_95CI.csv
